In [43]:
import os

import pandas as pd
import numpy as np

import regex as re
from datetime import datetime

In [44]:
data_path = "../data/in/"
output_path = "../data/out/"

regions = {"vor": "20241214-0617_gtfs_vor_2024", #vienna, lower austria, burgenland
           "ooevv": "20241212-0156_gtfs_ooevv_2024", #upper austria
           "esg": "20241203-0058_gtfs_esg_2024", #linz
           "verbundlinie": "20241217-0310_gtfs_verbundlinie_2024", #styria
           "kaernterlinien": "20241214-0253_gtfs_kaerntnerlinien_2024", #carinthia
           "salzburgverkehr": "20241217-0359_gtfs_salzburgverkehr_2024", #salzburg
           "vvt": "20241217-0436_gtfs_vvt_2024", #tyrol
           "vmobil": "20241212-0624_gtfs_vmobil_2024", #vorarlberg
           "obb": "GTFS_2024_obb"} #oebb maybe 20241217-0222_gtfs_evu_2024

# select region
state_name = "vor"
# select day for calculation in format YYYYMMDD in 2024
selected_day = 20240530

# stop categories
table = np.array([
    ["I", "I", "II", "III"],        # < 5 min
    ["I", "II", "III", "III"],      # 5 >= x <= 10
    ["II", "III", "IV", "IV"],      # 10 < x < 20
    ["III", "IV", "V", "V"],        # 20 >= x < 40
    ["IV", "V", "VI", "VI"],        # 40 >= x <= 60
    ["V", "VI", "VII", "VII"],      # 60 < x <= 120  
    ["", "VII", "VIII", "VIII"],    # 120 < x <= 210 
    ["", "", "", ""],               # > 210

])

# transport_category = ["Fernverkehr REX", 
#                       "S-Bahn / U-Bahn, Regionalbahn, Schnellbus, Lokalbahn", 
#                       "Straßenbahn, Metrobus, 0-Bus", 
#                       "Bus"]
route_type_translation = {0: 2, 1: 1, 2: 0, 3: 3, 11: 3, }


In [58]:
def lookup_category(interval, t_cat):
    if interval < 5:
        return table[0][t_cat]
    elif interval <= 10:
        return table[1][t_cat]
    elif interval < 20:
        return table[2][t_cat]
    elif interval < 40:
        return table[3][t_cat]
    elif interval <= 60:
        return table[4][t_cat]
    elif interval <= 120:
        return table[5][t_cat]
    elif interval <= 210:
        return table[6][t_cat]
    else:
        return table[7][t_cat]
    
def detect_route_type(trips_name, route_type):
    # TODO: also consider route_short/long_name ???
    if route_type == 2 and not pd.isna(trips_name):
        trips_name = trips_name.lower()
        if any(x in trips_name for x in ["rj", "rjx", "nj", "en", "ic", "ec", "ice", "ecb", "rex"]): #fernverkehr
            return 0
        else:
            return 1
    else:
        return route_type_translation[route_type]

In [54]:
# read in the data
path = f"{data_path}/{regions[state_name]}/"

stops = pd.read_csv(path + "/stops.txt", quotechar='"', sep=",")
stop_times = pd.read_csv(path + "/stop_times.txt", quotechar='"', sep=",")
trips = pd.read_csv(path + "/trips.txt", quotechar='"', sep=",")
routes = pd.read_csv(path + "/routes.txt", quotechar='"', sep=",")
calendar = pd.read_csv(path + "/calendar.txt", quotechar='"', sep=",")
calendar_dates = pd.read_csv(path + "/calendar_dates.txt", quotechar='"', sep=",")

print(stops.shape)
stops.head()

/var/folders/t9/cqlkdlt50h94khlrvmtm0zbc0000gn/T/ipykernel_31063/916577950.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(path + "/stop_times.txt", quotechar='"', sep=",")


(33185, 9)


/var/folders/t9/cqlkdlt50h94khlrvmtm0zbc0000gn/T/ipykernel_31063/916577950.py:6: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(path + "/trips.txt", quotechar='"', sep=",")


,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:41:10005:0:1,Sieggraben Gemeindeamt,47.650112,16.379279,1050.0,NaN,Pat:41:10005,Level 0,1
1,at:41:10005:0:2,Sieggraben Gemeindeamt,47.650099,16.379405,1050.0,NaN,Pat:41:10005,Level 0,2
2,at:41:10018:0:1,Antau Kleine Zeile,47.774540,16.475264,1051.0,NaN,Pat:41:10018,Level 0,1
3,at:41:10018:0:2,Antau Kleine Zeile,47.774450,16.475336,1051.0,NaN,Pat:41:10018,Level 0,2
4,at:41:10019:0:1,Stöttera Ost,47.770036,16.464475,1051.0,NaN,Pat:41:10019,Level 0,1


In [55]:
calendar_filtered = calendar[(calendar['start_date'] <= selected_day) & (calendar['end_date'] >= selected_day)]
calendar_dates_filtered = calendar_dates[calendar_dates["date"] == selected_day]

# find weekday of selected day
days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
date = datetime.fromisoformat(str(selected_day))
day_string = days[date.weekday()]

# only keep services that run on that weekday
calendar_filtered = calendar_filtered[calendar_filtered[day_string] == 1]

print(calendar_filtered.shape)
calendar_filtered.head()

(804, 10)


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
4,T0#101,1,1,1,1,1,0,0,20240304,20240630
7,T0#104,1,1,1,1,1,0,0,20231210,20241214
8,T0#105,1,1,1,1,1,0,0,20240201,20241214
10,T0#107,1,1,1,1,1,0,0,20240331,20241026
12,T0#109,1,1,1,1,1,0,0,20231210,20241214


In [56]:
# keep only valid trips
trips_filtered = trips[trips['service_id'].isin(calendar_filtered['service_id'])]
print(trips_filtered.shape)

trips_filtered = trips_filtered[trips_filtered["service_id"].isin(calendar_dates_filtered[calendar_dates_filtered["exception_type"] == 2]["service_id"])]
print(trips_filtered.shape)

trips_full = pd.concat([trips_filtered, trips[trips["service_id"].isin(calendar_dates_filtered[calendar_dates_filtered["exception_type"]==1]["service_id"])]])
print(trips_full.shape)
trips_full.head()

(86708, 8)
(83221, 8)
(105233, 8)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,at:vor:100:,T0#2,1.T0.70-100-j24-1.6.H,70-100-j24-1.6.H,St. Pölten Landhaus Klangturm,NaN,0,10536.0
1,at:vor:100:,T0#2,10.T0.70-100-j24-1.1.H,70-100-j24-1.1.H,St. Pölten Staudratgasse,NaN,0,10565.0
2,at:vor:100:,T0#2,100.T0.70-100-j24-1.3.R,70-100-j24-1.3.R,St. Pölten Hauptbahnhof,NaN,1,10581.0
3,at:vor:100:,T0#2,101.T0.70-100-j24-1.3.R,70-100-j24-1.3.R,St. Pölten Hauptbahnhof,NaN,1,10582.0
4,at:vor:100:,T0#2,102.T0.70-100-j24-1.3.R,70-100-j24-1.3.R,St. Pölten Hauptbahnhof,NaN,1,10583.0


In [57]:
# merge trips with routes information
routes_trips = pd.merge(trips_full, routes, on='route_id', how='left')

# translate route type
routes_trips['trip_short_name'] = routes_trips['trip_short_name'].astype('str')
routes_trips['rank'] = routes_trips.apply(lambda x: detect_route_type(x['trip_short_name'], x['route_type']), axis=1)

print(routes_trips.shape)
routes_trips.sort_values(by=['rank'])

(105233, 13)


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type,rank
32365,at:vor:3302:,T9#1,398.T9.21-U2-j24-4.7.R,21-U2-j24-4.7.R,Wien Seestadt,nan,1,NaN,4,U2,Seestadt - Schottentor,1,1
34902,at:vor:3304:,T8#5,57.T8.21-U4-j24-1.1.H,21-U4-j24-1.1.H,Wien Heiligenstadt,nan,0,NaN,4,U4,Hütteldorf - Heiligenstadt,1,1
34901,at:vor:3304:,T8#5,568.T8.21-U4-j24-1.2.R,21-U4-j24-1.2.R,Wien Hütteldorf,nan,1,NaN,4,U4,Hütteldorf - Heiligenstadt,1,1
34900,at:vor:3304:,T8#5,566.T8.21-U4-j24-1.2.R,21-U4-j24-1.2.R,Wien Hütteldorf,nan,1,NaN,4,U4,Hütteldorf - Heiligenstadt,1,1
34899,at:vor:3304:,T8#5,564.T8.21-U4-j24-1.2.R,21-U4-j24-1.2.R,Wien Hütteldorf,nan,1,NaN,4,U4,Hütteldorf - Heiligenstadt,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
51230,at:vor:3438:,T9#116,84583.T9.23-38A-j24-2.9.R,23-38A-j24-2.9.R,Wien Heiligenstadt,nan,1,NaN,4,38A,Heiligenstadt - Kahlenberg (-Leopoldsberg),3,3
51229,at:vor:3438:,T9#116,84582.T9.23-38A-j24-2.9.R,23-38A-j24-2.9.R,Wien Heiligenstadt,nan,1,NaN,4,38A,Heiligenstadt - Kahlenberg (-Leopoldsberg),3,3
51228,at:vor:3438:,T9#116,84581.T9.23-38A-j24-2.9.R,23-38A-j24-2.9.R,Wien Heiligenstadt,nan,1,NaN,4,38A,Heiligenstadt - Kahlenberg (-Leopoldsberg),3,3
51235,at:vor:3438:,T9#116,84588.T9.23-38A-j24-2.9.R,23-38A-j24-2.9.R,Wien Heiligenstadt,nan,1,NaN,4,38A,Heiligenstadt - Kahlenberg (-Leopoldsberg),3,3


In [59]:
# prepare stops and stop_times
stops_filtered = stops.copy()
stops_filtered['stop_id'] = stops_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)
# keep only stop entry for parent station, if no parent station is given, keep a stop entry
stops_parents = stops_filtered[stops_filtered['stop_id'].str.startswith('Pat')].copy()
stops_parents['stop_id'] = stops_parents['stop_id'].apply(lambda x: x if x[0] != 'P' else x[1:])
stops_filtered = stops_filtered[stops_filtered['stop_id'].str.startswith('at')]
stops_filtered = stops_filtered[~stops_filtered['stop_id'].isin(stops_parents['stop_id'])].drop_duplicates(subset=['stop_id'], keep='first')
# TODO: maybe filter out special stations e.g. obb_CP_80854 Wattens Sammelpunkt Bahnhofstraße MPREIS or Pat:42:99979_HoB

stops_filtered_final = pd.concat([stops_parents, stops_filtered])
print(stops_filtered_final.shape)

stop_times_filtered = stop_times.copy()
stop_times_filtered = stop_times_filtered[stop_times_filtered['departure_time'].between('06:00:00', '20:00:00')]
stop_times_filtered = stop_times_filtered[stop_times_filtered['stop_id'].str.startswith('at')]
# TODO: check if Parent station is in stop_times and keep those
stop_times_filtered['stop_id'] = stop_times_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)

# display(stop_times_df_obb_mod.head())
stops_filtered_final

(10875, 9)


,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
22311,at:41:10005,Sieggraben Gemeindeamt,47.649966,16.379351,NaN,1.0,NaN,NaN,NaN
22312,at:41:10018,Antau Kleine Zeile,47.774498,16.475309,NaN,1.0,NaN,NaN,NaN
22313,at:41:10019,Stöttera Ost,47.769928,16.464412,NaN,1.0,NaN,NaN,NaN
22314,at:41:10020,Zemendorf Volksschule,47.765448,16.454648,NaN,1.0,NaN,NaN,NaN
22315,at:41:10021,Zemendorf Wr. Neustädter Str.,47.762356,16.451638,NaN,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
21094,at:49:627,Wien Kagran U,48.243299,16.433762,100.0,NaN,NaN,Level 0,26
21516,at:49:752,Wien Lassallestraße,48.223834,16.402267,100.0,NaN,NaN,Level 0,11
21751,at:49:827,Wien Margaretengürtel,48.188865,16.342655,100.0,NaN,NaN,Level 0,60
22130,at:49:950,Wien Nußdorfer Straße,48.231500,16.353803,100.0,NaN,NaN,Level 0,6


In [60]:
stop_times_trips = pd.merge(stop_times_filtered, routes_trips, on='trip_id', how='inner')
print(stop_times_trips.shape)
stop_times_trips.head()

(1590066, 21)


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,route_id,...,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type,rank
0,10.T3.4-802-H-j24-1.4.R,12:45:00,12:45:00,at:43:70031,1,NaN,0,0,0.00,at:vor:1803:,...,4-802-H-j24-1.4.R,Alt-Nagelberg Bahnhof,R 3,1,NaN,81,WVB,Altnagelberg - Heidenreichstein Sondertarif,2,1
1,10.T3.4-802-H-j24-1.4.R,13:00:00,13:00:00,at:43:71984,2,NaN,0,0,4858.32,at:vor:1803:,...,4-802-H-j24-1.4.R,Alt-Nagelberg Bahnhof,R 3,1,NaN,81,WVB,Altnagelberg - Heidenreichstein Sondertarif,2,1
2,10.T3.4-802-H-j24-1.4.R,13:05:00,13:05:00,at:43:71985,3,NaN,0,0,6954.20,at:vor:1803:,...,4-802-H-j24-1.4.R,Alt-Nagelberg Bahnhof,R 3,1,NaN,81,WVB,Altnagelberg - Heidenreichstein Sondertarif,2,1
3,10.T3.4-802-H-j24-1.4.R,13:20:00,13:20:00,at:43:18450,4,NaN,0,0,11336.57,at:vor:1803:,...,4-802-H-j24-1.4.R,Alt-Nagelberg Bahnhof,R 3,1,NaN,81,WVB,Altnagelberg - Heidenreichstein Sondertarif,2,1
4,10.T3.4-802-H-j24-1.4.R,13:25:00,13:25:00,at:43:71982,5,NaN,0,0,12997.84,at:vor:1803:,...,4-802-H-j24-1.4.R,Alt-Nagelberg Bahnhof,R 3,1,NaN,81,WVB,Altnagelberg - Heidenreichstein Sondertarif,2,1


In [61]:
stop_times_grouped = stop_times_trips.groupby(['stop_id']).agg(rank=("rank", "min"), count=("rank", "count")).reset_index()

# TODO: maybe after merge with obb stations
# stop_times_grouped["interval"] = stop_times_grouped["count"].apply(lambda x: 840 / (x/2))
# stop_times_grouped["category"] = stop_times_grouped.apply(lambda x: lookup_category(x["interval"], x["rank"]), axis=1)

stop_times_grouped

,stop_id,rank,count
0,at:41:10005,3,131
1,at:41:10018,3,38
2,at:41:10019,3,39
3,at:41:10020,3,39
4,at:41:10021,3,39
...,...,...,...
10678,at:49:995,3,709
10679,at:49:996,2,328
10680,at:49:997,3,374
10681,at:49:998,3,169


In [62]:
stops_final = pd.merge(stops_filtered_final.drop(['zone_id', 'location_type', 'level_id', 'platform_code', 'parent_station'], axis=1), stop_times_grouped, on='stop_id', how='left')
stops_final

,stop_id,stop_name,stop_lat,stop_lon,rank,count
0,at:41:10005,Sieggraben Gemeindeamt,47.649966,16.379351,3.0,131.0
1,at:41:10018,Antau Kleine Zeile,47.774498,16.475309,3.0,38.0
2,at:41:10019,Stöttera Ost,47.769928,16.464412,3.0,39.0
3,at:41:10020,Zemendorf Volksschule,47.765448,16.454648,3.0,39.0
4,at:41:10021,Zemendorf Wr. Neustädter Str.,47.762356,16.451638,3.0,39.0
...,...,...,...,...,...,...
10870,at:49:627,Wien Kagran U,48.243299,16.433762,1.0,4437.0
10871,at:49:752,Wien Lassallestraße,48.223834,16.402267,3.0,808.0
10872,at:49:827,Wien Margaretengürtel,48.188865,16.342655,1.0,2619.0
10873,at:49:950,Wien Nußdorfer Straße,48.231500,16.353803,1.0,3306.0


In [63]:
# save to file
stops_final.to_csv(output_path + f"stops_{state_name}_{selected_day}.csv", index=False)